# TASS 2018: enfoques de clasificación de texto — representaciones y algoritmos

**Curso:** Procesamiento de Lenguaje Natural · Unidad 3 — Clasificación de texto  
**Institución:** Universidad Tecnológica de Bolívar  
**Basado en:** `bow_softmax_tass.ipynb` (Bag-of-Words + regresión softmax)  
**Referencias:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 4 — Naive Bayes, Text Classification, and Sentiment; borrador 2026: cap. 4 “Logistic Regression and Text Classification”, Apéndice B “Naive Bayes”, cap. 11 “Retrieval-Based Models” (TF-IDF)  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook es un **banco de comparación** para clasificar el sentimiento de tweets en español (TASS 2018). En una primera parte fija el clasificador (regresión logística) y compara seis formas de convertir texto en vectores: Bag-of-Words, BoW con n-gramas, TF-IDF, n-gramas de caracteres, *hashing trick* y embeddings Word2Vec promediados. En la segunda fija la representación (TF-IDF) y compara seis algoritmos: Naive Bayes, regresión logística, SVM lineal, árbol de decisión, Random Forest y k vecinos más cercanos. Al final reúne todos los resultados en una tabla y un gráfico.

**Objetivo:** entender cómo influyen por separado la **representación** y el **algoritmo** en el desempeño de un clasificador de texto, y contar con una referencia para elegir la combinación adecuada en un problema nuevo.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Finanzas y servicio al cliente — Elegir el modelo de intenciones de un chat

- **Problema:** un banco debe decidir qué representación y qué algoritmo llevar a producción para clasificar mensajes del chat.
- **Entrada:** mensajes del chat etiquetados con su intención.
- **Cómo se aplica la comparación de representaciones:**
  1. Se fija el protocolo: misma validación cruzada y mismas particiones para todas las variantes.
  2. Se comparan las seis representaciones con regresión logística y, después, los seis algoritmos con TF-IDF.
  3. Se registra el F1 y el tiempo de entrenamiento y predicción de cada combinación.
- **Salida:** tabla F1 × costo de cómputo.
- **Decisión:** se elige el modelo más simple cuyo F1 esté dentro del margen de error del mejor.

### Salud — Mensajes con errores de ortografía

- **Problema:** los pacientes escriben con errores (`kiero sita`, `no me funsiona la app`) y el clasificador de palabras falla.
- **Entrada:** mensajes informales del portal de pacientes.
- **Cómo se aplica la comparación de representaciones:**
  1. Se comparan BoW de palabras y n-gramas de caracteres, que comparten fragmentos aunque la palabra esté mal escrita (`cit`, `ita`).
  2. Se usa la misma validación cruzada para ambas.
  3. Se revisan los mensajes que solo acierta la representación por caracteres.
- **Salida:** F1 por representación y ejemplos de mensajes recuperados.
- **Decisión:** adoptar n-gramas de caracteres si la mejora es clara frente al costo adicional.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install numpy pandas scikit-learn imbalanced-learn gensim matplotlib seaborn nltk spacy requests
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `pathlib, collections.Counter, random` | Rutas, conteos y aleatoriedad del corpus de respaldo |
| `numpy, pandas, matplotlib, seaborn` | Cálculo, tablas y gráficas |
| `sklearn.feature_extraction.text` | `CountVectorizer`, `TfidfVectorizer`, `HashingVectorizer` |
| `sklearn (modelos)` | `MultinomialNB`, `LogisticRegression`, `LinearSVC`, `DecisionTreeClassifier`, `RandomForestClassifier`, `KNeighborsClassifier` |
| `sklearn (evaluación)` | Métricas, `ShuffleSplit`, `train_test_split`, `LabelEncoder` |
| `imblearn.over_sampling.RandomOverSampler` | Sobremuestreo de clases minoritarias |
| `gensim.models.Word2Vec` | Embeddings entrenados sobre el propio corpus (sección 9.6) |
| `logic.text_processing.TextProcessing` | Normalizador de tweets del curso |

**Datos:** `data/tass/tass2018_es_train.csv` y `tass2018_es_test.csv`. Si no están, el notebook genera un corpus sintético solo de demostración y lo avisa.

## Marco metodológico: CRISP-ML(Q)

Este notebook sigue **CRISP-ML(Q)** (*Cross-Industry Standard Process for the development of Machine Learning applications with Quality assurance*; Studer et al., 2021). El proceso tiene seis fases, y en cada una se identifican los riesgos de calidad y cómo se mitigan. Las fases aparecen marcadas con 🔷 a lo largo del notebook.

| Fase CRISP-ML(Q) | Dónde se aborda en este notebook | Aseguramiento de calidad |
|------------------|----------------------------------|--------------------------|
| 1. Entendimiento del negocio y de los datos | Secciones 2, 4 y 6: etiquetas de TASS, carga y distribución de clases | Aviso explícito si se usa el corpus sintético en lugar del real |
| 2. Ingeniería de datos | Secciones 5, 7 y 9: normalización, sobremuestreo del texto y seis representaciones | Cada vectorizador se ajusta solo con entrenamiento |
| 3. Ingeniería del modelo | Secciones 8 y 10: función de experimentos y seis algoritmos | Mismos datos y misma representación para comparar algoritmos |
| 4. Aseguramiento de calidad del modelo | Secciones 10.2, 11 y 12: validación cruzada sin fuga, reporte por clase y comparación final | El sobremuestreo y el vectorizador se ajustan **dentro** de cada partición |
| 5. Despliegue | Al final (fuera de alcance) | — |
| 6. Monitoreo y mantenimiento | Al final (fuera de alcance) | — |

Nota: la tabla final compara todos los enfoques **en el conjunto de prueba**. Sirve para aprender, pero elegir el “mejor” modelo con el mismo test que se reporta es optimista; para una selección rigurosa usa validación cruzada (como en la sección 10.2 o en `bow_lexico_tass.ipynb`).

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | Se corrige un riesgo de **fuga de datos** en la validación cruzada (sección 10.2) y se advierte del sesgo de elegir modelos con el conjunto de prueba. |
| **IEEE 7003-2024** — Consideraciones sobre sesgo algorítmico | Se mide el desbalance de clases, se corrige con sobremuestreo y se reportan métricas macro y por clase. |
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | Se comparan modelos de distinta transparencia: regresión logística y árboles son interpretables; k-NN y Random Forest lo son menos. |
| **IEEE 2894-2024** — Guía de marco arquitectónico para IA explicable (XAI) | El reporte por clase (sección 11) muestra dónde se equivoca el modelo, base para explicar sus decisiones. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | Todo el código y las librerías son abiertas; el corpus sintético de respaldo permite ejecutar el notebook sin el corpus real. |

## Objetivos de aprendizaje

Este notebook extiende `bow_softmax_tass.ipynb` (Bag-of-Words + softmax) con los distintos enfoques vistos en la unidad de clasificación de texto. Al terminar podrás:

1. Cargar y preprocesar el corpus **TASS 2018** (tweets en español, 4 clases de polaridad)
2. Comparar **cinco técnicas de extracción de features**: Bag-of-Words, BoW con n-gramas, TF-IDF, n-gramas de caracteres y el Hashing Trick
3. Comparar **seis algoritmos de clasificación clásicos**: Naive Bayes, Logistic Regression (Softmax), SVM lineal, Decision Tree, Random Forest y K-Nearest Neighbors
4. Manejar el **desbalance de clases** con `RandomOverSampler`, preservando el texto crudo para reutilizarlo en cada técnica
5. Construir una **tabla y gráfico comparativo** de accuracy, precision, recall y F1 (macro) entre todos los enfoques

## Dataset — TASS 2018

[TASS](http://tass.sepln.org/) (Taller de Análisis Semántico en la SEPLN) es una campaña de evaluación de análisis de sentimiento en español. La edición 2018 usa tweets etiquetados con cuatro clases de polaridad.

| Etiqueta | Significado |
|---|---|
| **P** | Sentimiento positivo |
| **N** | Sentimiento negativo |
| **NEU** | Neutral: el tweet expresa a la vez polaridad positiva y negativa |
| **NONE** | Sin sentimiento: el tweet es informativo u objetivo |

> **Nota sobre los datos:** el corpus real de TASS requiere registro previo en `tass.sepln.org` y no se redistribuye aquí. Si tienes los CSV oficiales, colócalos en `data/tass/tass2018_es_train.csv` y `data/tass/tass2018_es_test.csv` con las columnas `content` y `sentiment/polarity/value`, y el notebook los usará automáticamente. **Si no los tienes**, la Sección 4 genera un corpus sintético de demostración con la misma estructura y el mismo desbalance de clases, para que todo el notebook corra de principio a fin sin configuración adicional.

---

## 1. Configuración del entorno

In [ ]:
from pathlib import Path
from collections import Counter
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.feature_extraction.text import (
    CountVectorizer, TfidfVectorizer, HashingVectorizer,
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import ShuffleSplit, train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler

sns.set_theme(style="whitegrid")

# Normalizador de tweets del curso (carpeta logic/ junto a este notebook)
from logic.text_processing import TextProcessing

DIR_DATA = Path.cwd() / 'data' / 'tass'
print(f'Directorio de datos: {DIR_DATA}')

> 🔷 **CRISP-ML(Q) · Fase 1 — Entendimiento del negocio y de los datos.** Etiquetas, carga y distribución de clases.

---

## 2. El conjunto de datos TASS 2018

[TASS](http://tass.sepln.org/) (Taller de Análisis Semántico en la SEPLN) es una campaña de evaluación de análisis de sentimiento en español. La edición 2018 usa tweets etiquetados con cuatro clases de polaridad.

| Etiqueta | Significado |
|---|---|
| **P** | Sentimiento positivo |
| **N** | Sentimiento negativo |
| **NEU** | Neutral: el tweet expresa a la vez polaridad positiva y negativa |
| **NONE** | Sin sentimiento: el tweet es informativo u objetivo |

---

## 3. Inicializar utilidades de preprocesamiento

In [ ]:
tp = TextProcessing()   # normalizador de tweets: minúsculas, URL, menciones, hashtags, emojis, puntuación
le = LabelEncoder()     # convierte las etiquetas de texto (P, N, NEU, NONE) en enteros

---

## 4. Cargar entrenamiento y prueba

Se intenta cargar el corpus real de TASS 2018 desde `DIR_DATA`. Si los archivos no existen, se genera un **corpus sintético de demostración** con la misma estructura de columnas (`content`, `sentiment/polarity/value`) y un desbalance de clases similar al real (NEU/NONE más frecuentes que P/N), para que el resto del notebook sea reproducible sin depender del acceso restringido a TASS.

In [ ]:
def _build_synthetic_tass(seed: int = 42) -> pd.DataFrame:
    """Genera un corpus sintético de tweets en español con la misma forma que TASS 2018.

    No sustituye al corpus real: es solo un conjunto de demostración para que el
    notebook corra de extremo a extremo cuando `tass2018_es_*.csv` no está disponible.
    """
    rng = random.Random(seed)

    topics = [
        "el restaurante", "la película", "el concierto", "el curso de NLP",
        "la conferencia", "el servicio al cliente", "la aplicación", "el hotel",
        "el partido de fútbol", "la clase de historia", "el libro nuevo",
        "el producto", "el evento", "la ciudad", "el vuelo", "la reunión",
    ]

    templates = {
        "P": [
            "me encanta {t}, es increíble",
            "qué bueno estuvo {t}, lo recomiendo totalmente",
            "{t} superó mis expectativas, gracias",
            "excelente {t}, sin duda vuelvo pronto",
            "el mejor {t} que he probado en mi vida",
            "feliz con {t}, todo salió perfecto",
            "{t} es una maravilla, diez de diez",
            "disfruté muchísimo {t}, una experiencia genial",
        ],
        "N": [
            "pésimo servicio en {t}, no vuelvo más",
            "qué decepción {t}, esperaba mucho más",
            "{t} fue un desastre total",
            "no me gustó nada {t}, una pérdida de tiempo",
            "terrible experiencia con {t}, muy mal",
            "{t} me arruinó el día, horrible",
            "nunca había visto algo tan malo como {t}",
            "evitar {t} a toda costa, no lo recomiendo",
        ],
        "NEU": [
            "{t} estuvo bien, nada especial",
            "fui a {t}, ni bueno ni malo",
            "{t} cumple lo básico, sin más",
            "{t} de hoy fue normal",
            "asistí a {t}, todo tranquilo",
            "{t} funcionó como se esperaba",
        ],
        "NONE": [
            "mañana hay información sobre {t} en la universidad",
            "el horario de {t} cambió para la próxima semana",
            "comparto el enlace con los detalles de {t}",
            "recordatorio: la entrega relacionada con {t} vence el viernes",
            "nueva actualización disponible sobre {t}",
            "el equipo encargado de {t} se reúne el lunes",
        ],
    }

    # Desbalance intencional, similar al real: NEU y NONE más frecuentes que P y N
    n_per_class = {"P": 45, "N": 45, "NEU": 70, "NONE": 70}

    rows = []
    for label, count in n_per_class.items():
        combos = [(tmpl, topic) for tmpl in templates[label] for topic in topics]
        rng.shuffle(combos)
        for tmpl, topic in combos[:count]:
            rows.append({"content": tmpl.format(t=topic), "sentiment/polarity/value": label})

    df = pd.DataFrame(rows)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    # Ruido de etiquetas (~8%): evita que el corpus sea trivialmente separable,
    # para que la comparación final entre técnicas/algoritmos sea más realista.
    all_labels = list(templates.keys())
    n_noisy = int(0.08 * len(df))
    noisy_idx = rng.sample(range(len(df)), n_noisy)
    for i in noisy_idx:
        current = df.loc[i, 'sentiment/polarity/value']
        alt_labels = [l for l in all_labels if l != current]
        df.loc[i, 'sentiment/polarity/value'] = rng.choice(alt_labels)

    return df


def load_tass(dir_data: Path):
    train_path = dir_data / 'tass2018_es_train.csv'
    test_path = dir_data / 'tass2018_es_test.csv'

    if train_path.exists() and test_path.exists():
        print('Cargando corpus real de TASS 2018 desde', dir_data)
        return pd.read_csv(train_path, sep=','), pd.read_csv(test_path, sep=',')

    print('CSV de TASS no encontrados en', dir_data)
    print('→ Generando corpus sintético de demostración (no reemplaza al corpus real).')
    full = _build_synthetic_tass(seed=42)
    train_df, test_df = train_test_split(
        full, test_size=0.25, random_state=42,
        stratify=full['sentiment/polarity/value'],
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


# Carga TASS real o, si falta, el corpus sintético
data_train, data_test = load_tass(DIR_DATA)

print(f'\nEjemplos de entrenamiento: {len(data_train):,}')
print(f'Ejemplos de prueba:        {len(data_test):,}')
print(f'\nColumnas: {data_train.columns.tolist()}')
print(data_train.head(3).to_string())

> **Interpretación:** el CSV (real o sintético) contiene el texto del tweet (`content`) y la etiqueta de polaridad. Revisa el mensaje impreso: si dice que se generó el **corpus sintético**, los resultados de este notebook solo ilustran la mecánica y no son comparables con TASS real.

> 🔷 **CRISP-ML(Q) · Fase 2 — Ingeniería de datos.** Normalización, balanceo y seis representaciones del texto.

---

## 5. Preprocesamiento de texto y codificación de etiquetas

`TextProcessing.transformer()` normaliza cada tweet: elimina tildes, pasa a minúsculas, reemplaza emojis, URL, menciones (@) y hashtags (#) por marcadores, elimina la mayoría de signos de puntuación y colapsa los espacios. Esto reduce el tamaño del vocabulario y mejora la generalización.

In [ ]:
# Aplica el normalizador del curso a cada tweet (cadena vacía si el tweet queda vacío)
texts_train = [tp.transformer(row) or '' for row in data_train['content'].tolist()]
labels_train = le.fit_transform(data_train['sentiment/polarity/value'].tolist())

texts_test = [tp.transformer(row) or '' for row in data_test['content'].tolist()]
labels_test = le.transform(data_test['sentiment/polarity/value'].tolist())

print(f'Clases: {le.classes_.tolist()}')
print(f'Distribución en entrenamiento: {sorted(Counter(labels_train).items())}')
print(f'Distribución en prueba:        {sorted(Counter(labels_test).items())}')

> **Interpretación:** `LabelEncoder` convierte las etiquetas de texto a enteros en orden alfabético (N=0, NEU=1, NONE=2, P=3 por defecto). La distribución revela el desbalance típico de los datos de Twitter: en TASS dominan N y P, y NEU y NONE son minoritarias. `RandomOverSampler` (Sección 7) corrige esto.

> **Importante:** aquí mantenemos `texts_train` / `texts_test` como **listas de texto crudo**, sin vectorizar todavía. Esto permite reutilizar el mismo texto preprocesado con cada una de las técnicas de extracción de features de la Sección 6, en lugar de vectorizar una sola vez y perder el texto original.

---

## 6. Análisis visual de la distribución de clases

In [ ]:
# Barras con el número de tweets por clase en entrenamiento y prueba
fig, ax = plt.subplots(figsize=(7, 4))
class_names = le.classes_
train_counts = [Counter(labels_train)[i] for i in range(len(class_names))]
test_counts = [Counter(labels_test)[i] for i in range(len(class_names))]

x = np.arange(len(class_names))
width = 0.35
ax.bar(x - width / 2, train_counts, width, label='Train', color='#098559')
ax.bar(x + width / 2, test_counts, width, label='Test', color='#2CB5E0')
ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.set_ylabel('Número de tweets')
ax.set_title('Distribución de clases — TASS 2018')
ax.legend()
plt.tight_layout()
plt.show()

---

## 7. Manejo del desbalance de clases — `RandomOverSampler`

`RandomOverSampler` duplica ejemplos de las clases minoritarias hasta igualar el conteo de la clase mayoritaria. El workshop original lo aplicaba **después** de vectorizar (sobre la matriz BoW), lo cual amarra el sobremuestreo a una sola representación. Aquí lo aplicamos sobre los **índices** del texto crudo, para poder reutilizar el mismo conjunto balanceado con cualquier técnica de extracción de features de la Sección 8.

In [ ]:
# Sobremuestrea índices (no matrices) para reutilizar el mismo texto balanceado con cualquier representación
ros = RandomOverSampler(random_state=1000)
idx = np.arange(len(texts_train)).reshape(-1, 1)
idx_resampled, labels_train_bal = ros.fit_resample(idx, labels_train)
texts_train_bal = [texts_train[i] for i in idx_resampled.flatten()]

print('Antes del sobremuestreo  — train:', sorted(Counter(labels_train).items()))
print('Después del sobremuestreo — train:', sorted(Counter(labels_train_bal).items()))

> **Interpretación:** tras el sobremuestreo todas las clases quedan con el mismo conteo, evitando que el clasificador se sesgue hacia la clase mayoritaria. El sobremuestreo duplica ejemplos existentes sin agregar información nueva — para mejores resultados considera SMOTE (interpola entre ejemplos minoritarios en lugar de duplicarlos).

> 🔷 **CRISP-ML(Q) · Fase 3 — Ingeniería del modelo.** Función de experimentos y comparación de representaciones y algoritmos.

---

## 8. Función auxiliar — `run_experiment`

Para comparar muchas combinaciones de *representación × algoritmo* sin repetir código, definimos una función que entrena, evalúa sobre el conjunto de prueba y guarda las métricas macro (accuracy, precision, recall, F1) en una lista compartida `results`.

In [ ]:
results = []  # cada entrada: {'Enfoque': ..., 'Accuracy': ..., 'Precision': ..., 'Recall': ..., 'F1-score': ...}


def run_experiment(name, X_train, y_train, X_test, y_test, clf, verbose=True):
    # Entrena `clf`, evalua sobre test y registra metricas macro en `results`.
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    row = {
        'Enfoque': name,
        'Accuracy': round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test, y_pred, average='macro', zero_division=0) * 100, 2),
        'Recall': round(recall_score(y_test, y_pred, average='macro', zero_division=0) * 100, 2),
        'F1-score': round(f1_score(y_test, y_pred, average='macro', zero_division=0) * 100, 2),
    }
    results.append(row)
    if verbose:
        print(f"{name:35s} -> Accuracy {row['Accuracy']:5.2f}%  |  F1 (macro) {row['F1-score']:5.2f}%")
    return clf, row

---

## 9. Técnicas de extracción de features

Fijamos el clasificador (**Logistic Regression**) y variamos la representación, para aislar el efecto de cada técnica de extracción de features sobre el mismo problema. Cada vectorizador se ajusta (`fit`) **solo** con `texts_train_bal` para evitar fuga de información (*data leakage*) del conjunto de prueba.

### 9.1 Bag-of-Words (unigramas) — `CountVectorizer`

Cada documento se representa como un vector de conteos sobre el vocabulario global: la posición i cuenta cuántas veces aparece la palabra i. Es la representación más simple y una línea base competitiva.

In [ ]:
# BoW de unigramas: el vectorizador se ajusta solo con el entrenamiento balanceado
bow_uni = CountVectorizer(analyzer='word', ngram_range=(1, 1))
X_train_bow_uni = bow_uni.fit_transform(texts_train_bal)
X_test_bow_uni = bow_uni.transform(texts_test)
print(f'Vocabulario: {len(bow_uni.vocabulary_):,}  |  Train: {X_train_bow_uni.shape}  |  Test: {X_test_bow_uni.shape}')

# Entrena regresión logística y evalúa en prueba
run_experiment('BoW unigramas + LogReg', X_train_bow_uni, labels_train_bal, X_test_bow_uni, labels_test,
               LogisticRegression(solver='lbfgs', C=10, max_iter=1000))

### 9.2 Bag-of-Words con n-gramas (1–3) — `CountVectorizer`

Incluir bigramas y trigramas captura patrones de orden local que los unigramas no distinguen (p. ej. "no me gusta" vs. "me gusta").

In [ ]:
# BoW con unigramas, bigramas y trigramas
bow_ngram = CountVectorizer(analyzer='word', ngram_range=(1, 3))
X_train_bow_ngram = bow_ngram.fit_transform(texts_train_bal)
X_test_bow_ngram = bow_ngram.transform(texts_test)

# Dispersión = proporción de celdas en cero
density = X_train_bow_ngram.nnz / (X_train_bow_ngram.shape[0] * X_train_bow_ngram.shape[1])
print(f'Vocabulario: {len(bow_ngram.vocabulary_):,}  |  Train: {X_train_bow_ngram.shape}  |  Sparsity: {1 - density:.4%}')

run_experiment('BoW n-gramas (1-3) + LogReg', X_train_bow_ngram, labels_train_bal, X_test_bow_ngram, labels_test,
               LogisticRegression(solver='lbfgs', C=10, max_iter=1000))

> **Interpretación:** incluir trigramas amplía el vocabulario significativamente (a menudo 10×–100× frente a solo unigramas). La sparsity es muy alta porque cada tweet es corto (15–30 tokens) frente a un vocabulario combinado de miles de n-gramas.

### 9.3 TF-IDF — `TfidfVectorizer`

Pondera cada término por su frecuencia en el documento (TF) y penaliza los términos frecuentes en todo el corpus (IDF), resaltando las palabras más distintivas de cada tweet.

In [ ]:
# TF-IDF con unigramas y bigramas (representación usada en la sección 10)
tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(texts_train_bal)
X_test_tfidf = tfidf.transform(texts_test)
print(f'Vocabulario: {len(tfidf.vocabulary_):,}  |  Train: {X_train_tfidf.shape}')

run_experiment('TF-IDF (1-2) + LogReg', X_train_tfidf, labels_train_bal, X_test_tfidf, labels_test,
               LogisticRegression(solver='lbfgs', C=10, max_iter=1000))

### 9.4 N-gramas de caracteres — `CountVectorizer(analyzer='char_wb')`

En vez de dividir por palabras, extrae subsecuencias de caracteres dentro de los límites de cada palabra. Es más robusto a errores tipográficos, alargamientos ("buenoooo") y a la morfología rica del español (conjugaciones, sufijos).

In [ ]:
# n-gramas de 3 a 5 caracteres dentro de cada palabra ('char_wb')
char_ngram = CountVectorizer(analyzer='char_wb', ngram_range=(3, 5))
X_train_char = char_ngram.fit_transform(texts_train_bal)
X_test_char = char_ngram.transform(texts_test)
print(f'Vocabulario (n-gramas de caracteres): {len(char_ngram.vocabulary_):,}  |  Train: {X_train_char.shape}')

run_experiment('N-gramas de caracteres (3-5) + LogReg', X_train_char, labels_train_bal, X_test_char, labels_test,
               LogisticRegression(solver='lbfgs', C=10, max_iter=1000))

### 9.5 Hashing Trick — `HashingVectorizer`

En vez de construir un vocabulario explícito, aplica una función hash a cada término. Produce un vector de tamaño **fijo** (`n_features`), sin necesitar memoria adicional por cada palabra nueva — útil para streams de texto o vocabularios muy grandes. Trade-off: pueden ocurrir colisiones (dos términos distintos caen en la misma posición).

In [ ]:
# Vector de tamaño fijo 2^12 = 4096 sin vocabulario; alternate_sign=False deja conteos no negativos
hashing = HashingVectorizer(n_features=2**12, alternate_sign=False)
X_train_hash = hashing.transform(texts_train_bal)
X_test_hash = hashing.transform(texts_test)
print(f'Dimensión fija: {X_train_hash.shape}  (no depende del tamaño del vocabulario)')

run_experiment('Hashing Trick (2^12) + LogReg', X_train_hash, labels_train_bal, X_test_hash, labels_test,
               LogisticRegression(solver='lbfgs', C=10, max_iter=1000))

### 9.6 (Opcional) Embeddings promediados — Word2Vec entrenado en el propio corpus

Promediar los embeddings de las palabras de cada tweet produce un vector **denso** de baja dimensión que captura semántica, a costa de perder el orden. Aquí entrenamos un Word2Vec pequeño **sobre el propio corpus** (no requiere descargar un modelo preentrenado, así que corre sin conexión a internet) — solo con fines ilustrativos; con un corpus tan chico los vectores no serán semánticamente ricos. Para un ejemplo con vectores preentrenados de calidad, ver el notebook de *Text Representation* (GloVe / Word2Vec).

In [ ]:
from gensim.models import Word2Vec

# Tokens por espacios para entrenar Word2Vec con el propio corpus
tokenized_train = [t.split() for t in texts_train_bal]
tokenized_test = [t.split() for t in texts_test]

# Skip-gram (sg=1) de 50 dimensiones
w2v = Word2Vec(sentences=tokenized_train, vector_size=50, window=3, min_count=1, sg=1, seed=42)


# Vector del tweet = promedio de los vectores de sus palabras (ceros si ninguna es conocida)
def average_embedding(tokens, model, dim):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)


X_train_emb = np.array([average_embedding(t, w2v, 50) for t in tokenized_train])
X_test_emb = np.array([average_embedding(t, w2v, 50) for t in tokenized_test])
print(f'Embeddings promediados — Train: {X_train_emb.shape}  (vector denso, ya no disperso)')

run_experiment('Embeddings promediados (Word2Vec propio) + LogReg', X_train_emb, labels_train_bal, X_test_emb, labels_test,
               LogisticRegression(solver='lbfgs', C=10, max_iter=1000))

> **Interpretación:** al ser un vector denso de solo 50 dimensiones (frente a miles de posiciones dispersas de BoW/TF-IDF), el desempeño con un Word2Vec entrenado sobre un corpus tan pequeño suele ser más bajo — necesita mucho más texto para aprender relaciones semánticas útiles. Con un modelo preentrenado sobre un corpus masivo (GloVe, FastText), esta misma técnica normalmente mejora bastante frente a esta demostración.

---

## 10. Algoritmos de clasificación clásicos

Ahora fijamos la representación (**TF-IDF, n-gramas 1–2** — la más sólida de la sección anterior) y variamos el **algoritmo**, para aislar el efecto del clasificador sobre la misma representación.

### 10.1 Naive Bayes — `MultinomialNB`

Clasificador generativo: modela cómo se genera el documento dado cada clase, asumiendo independencia condicional entre palabras. Extremadamente rápido de entrenar y una línea base sólida con poco dato.

In [ ]:
# Naive Bayes multinomial con suavizado de Laplace (alpha=1)
run_experiment('TF-IDF + Naive Bayes', X_train_tfidf, labels_train_bal, X_test_tfidf, labels_test,
               MultinomialNB(alpha=1.0))

### 10.2 Logistic Regression (Softmax) — validación cruzada `ShuffleSplit`

Este es el clasificador del workshop original. `LogisticRegression(solver='lbfgs')` con más de dos clases usa softmax internamente. `C=10` implica poca regularización L2 (el modelo se ajusta de cerca a los datos de entrenamiento). A diferencia de los demás algoritmos de esta sección, aquí se hace una validación cruzada de 10 particiones antes de evaluar en el conjunto de prueba.

> ⚠️ **Corrección de fuga de datos.** La versión anterior hacía la validación cruzada sobre el entrenamiento **ya sobremuestreado** y vectorizado: copias del mismo tweet quedaban en entrenamiento y validación, inflando las métricas. Ahora, en cada partición, el sobremuestreo y el TF-IDF se ajustan **solo** con la parte de entrenamiento, y la validación usa tweets originales.

In [ ]:
# C=10 → poca regularización L2; bajar a C=0.1 si el modelo sobreajusta
softmax = LogisticRegression(solver='lbfgs', C=10, max_iter=1000)

# 10 particiones aleatorias 75/25 sobre el texto ORIGINAL (sin duplicados); random_state fija la reproducibilidad
k_fold = ShuffleSplit(n_splits=10, test_size=0.25, random_state=42)

accuracy_scores, recall_scores, precision_scores, f1_scores = [], [], [], []
textos = np.array(texts_train, dtype=object)

for fold, (train_idx, val_idx) in enumerate(k_fold.split(textos, labels_train)):
    # 1) Sobremuestreo solo de la parte de entrenamiento de la partición
    idx_bal, y_fold_train = RandomOverSampler(random_state=1000).fit_resample(train_idx.reshape(-1, 1), labels_train[train_idx])
    # 2) TF-IDF ajustado solo con esa parte, y aplicado a la validación
    vec = TfidfVectorizer(analyzer='word', ngram_range=(1, 2))
    x_fold_train = vec.fit_transform(textos[idx_bal.ravel()])
    x_fold_val = vec.transform(textos[val_idx])
    y_fold_val = labels_train[val_idx]

    softmax.fit(x_fold_train, y_fold_train)
    y_pred_val = softmax.predict(x_fold_val)

    accuracy_scores.append(accuracy_score(y_fold_val, y_pred_val))
    recall_scores.append(recall_score(y_fold_val, y_pred_val, average='macro', zero_division=0))
    precision_scores.append(precision_score(y_fold_val, y_pred_val, average='macro', zero_division=0))
    f1_scores.append(f1_score(y_fold_val, y_pred_val, average='macro', zero_division=0))

print(f'Promedio sobre {k_fold.n_splits} particiones (macro):')
print(f'  Accuracy:  {np.mean(accuracy_scores) * 100:.2f}%')
print(f'  Precisión: {np.mean(precision_scores) * 100:.2f}%')
print(f'  Recall:    {np.mean(recall_scores) * 100:.2f}%')
print(f'  F1:        {np.mean(f1_scores) * 100:.2f}% ± {np.std(f1_scores) * 100:.2f}')

# Entrenar en todo el train balanceado y evaluar en test (para la tabla comparativa)
run_experiment('TF-IDF + Logistic Regression (Softmax)', X_train_tfidf, labels_train_bal, X_test_tfidf, labels_test,
               softmax)

> **Interpretación:** las métricas de CV estiman el desempeño esperado antes de tocar el conjunto de prueba. Si el F1 de test es mucho menor que el de CV, es señal de sobreajuste — considera bajar `C` (más regularización) o reducir el rango de n-gramas.

### 10.3 SVM lineal — `LinearSVC`

Busca el hiperplano que separa las clases con el margen máximo. Con kernel lineal es especialmente competitivo en features dispersas de alta dimensión como TF-IDF.

In [ ]:
# SVM lineal: busca el hiperplano de margen máximo
run_experiment('TF-IDF + SVM lineal', X_train_tfidf, labels_train_bal, X_test_tfidf, labels_test,
               LinearSVC(C=1.0, max_iter=5000))

### 10.4 Decision Tree

Particiona el espacio de features con reglas simples aprendidas automáticamente. Interpretable, pero propenso a sobreajustar por sí solo.

In [ ]:
# Árbol limitado a profundidad 20 para contener el sobreajuste
run_experiment('TF-IDF + Decision Tree', X_train_tfidf, labels_train_bal, X_test_tfidf, labels_test,
               DecisionTreeClassifier(max_depth=20, random_state=42))

### 10.5 Random Forest

Entrena muchos árboles sobre submuestras distintas de datos y features, y promedia sus predicciones — mitiga el sobreajuste de un árbol individual.

In [ ]:
# 200 árboles entrenados con submuestras de datos y de características
run_experiment('TF-IDF + Random Forest', X_train_tfidf, labels_train_bal, X_test_tfidf, labels_test,
               RandomForestClassifier(n_estimators=200, random_state=42))

### 10.6 K-Nearest Neighbors

Clasifica un documento nuevo según la clase mayoritaria entre sus k documentos más similares, sin fase de entrenamiento explícita.

In [ ]:
# k-NN con distancia coseno, adecuada para vectores TF-IDF
run_experiment('TF-IDF + K-NN (k=5)', X_train_tfidf, labels_train_bal, X_test_tfidf, labels_test,
               KNeighborsClassifier(n_neighbors=5, metric='cosine'))

> 🔷 **CRISP-ML(Q) · Fase 4 — Aseguramiento de calidad del modelo.** Reporte por clase y comparación final de todos los enfoques.

---

## 11. Reporte detallado del mejor modelo

`classification_report` desglosa precision, recall y F1 **por clase** (no solo el promedio macro) — útil para ver, por ejemplo, si el modelo confunde sistemáticamente NEU con NONE.

In [ ]:
best_row = max(results, key=lambda r: r['F1-score'])
print(f"Mejor enfoque por F1 (macro): {best_row['Enfoque']}  ->  F1 = {best_row['F1-score']}%\n")

# Se recalcula con TF-IDF + Logistic Regression como referencia legible; cambia el enfoque si prefieres otro
softmax_final = LogisticRegression(solver='lbfgs', C=10, max_iter=1000)
softmax_final.fit(X_train_tfidf, labels_train_bal)
y_pred_final = softmax_final.predict(X_test_tfidf)

print(classification_report(labels_test, y_pred_final, target_names=le.classes_, zero_division=0))

---

## 12. Comparación final — todos los enfoques

Consolidamos `results` (una fila por cada combinación de *representación × algoritmo* evaluada arriba) en una tabla y un gráfico de barras ordenado por F1 (macro).

In [ ]:
# Todos los experimentos ordenados por F1 macro en prueba
results_df = pd.DataFrame(results).sort_values('F1-score', ascending=False).reset_index(drop=True)
results_df

In [ ]:
# Barras horizontales de F1 macro por enfoque
fig, ax = plt.subplots(figsize=(9, 5))
order = results_df.sort_values('F1-score')
bars = ax.barh(order['Enfoque'], order['F1-score'], color='#098559')
ax.set_xlabel('F1-score (macro, %)')
ax.set_title('Comparación de enfoques — TASS 2018')
ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

> **Interpretación:** si se usó el corpus sintético, estas cifras solo ilustran la mecánica del *pipeline*: sus patrones son fáciles de separar por diseño. Con el corpus real de TASS 2018, la tabla sí sirve como comparación experimental, con dos precauciones: (1) se compara en el conjunto de prueba, así que la diferencia entre los primeros puestos puede ser ruido (ver el *bootstrap* pareado de `bow_lexico_tass.ipynb`); (2) el archivo de prueba de este repositorio tiene 506 tweets, que no es la partición de prueba oficial de TASS 2018, así que las cifras no son directamente comparables con las del *Overview* de la competencia.

---

## 13. Extensiones sugeridas

1. **Embeddings preentrenados**: sustituir el Word2Vec entrenado en el propio corpus por FastText (`cc.es.300.bin`) o por embeddings contextuales (BERT multilingüe) — ver el notebook de *Text Representation*.
2. **SMOTE en vez de RandomOverSampler**: genera ejemplos minoritarios sintéticos por interpolación en el espacio de features, en lugar de duplicar ejemplos existentes.
3. **Búsqueda de hiperparámetros**: usar `GridSearchCV` o `RandomizedSearchCV` para afinar `C` (Logistic Regression/SVM), `max_depth` (árboles) o `n_neighbors` (KNN).
4. **Análisis de errores**: inspeccionar los tweets mal clasificados por clase — NEU y NONE suelen ser las más difíciles de distinguir entre sí.
5. **Fine-tuning de un modelo preentrenado**: ajustar BERT multilingüe (`bert-base-multilingual-cased`) sobre TASS suele superar claramente a todos los enfoques clásicos de este notebook, al costo de mayor tiempo de entrenamiento y necesidad de GPU.
6. **Adaptación de dominio**: reentrenar sobre un dominio específico (reseñas de hoteles, tweets políticos) para mejorar el desempeño en esa aplicación concreta.

---

> 🔷 **CRISP-ML(Q) · Fases 5 y 6 — Despliegue, monitoreo y mantenimiento.** Están fuera del alcance de este notebook académico, pero así se abordarían:
>
> - **Despliegue:** elegir el enfoque por validación cruzada, empaquetar vectorizador y modelo en un `Pipeline` y guardarlo con `joblib`.
> - **Monitoreo y mantenimiento:** medir semanalmente la distribución de clases predichas y el porcentaje de términos fuera del vocabulario; con *hashing* este último no se puede medir directamente, lo que es un costo de esa técnica.